# MLB Game Winner Model — v2

Training workflow using **per-game batting logs** from Baseball Reference.
Rolling pre-game stats (last  games) are computed so that
no future data leaks into any game's feature vector.

**Features (15 total):**
- Home team: , , , , , ,  (7)
- Away team: same (7)
-  = 1

**Pipeline:**
1. Fetch per-game batting logs from BRef via 
2. Compute rolling pre-game stats via 
3. Build feature vectors for home games across multiple seasons
4. Train GradientBoosting classifier (home win = 1)
5. Evaluate and save to 


## 1. Setup

In [ ]:
import os, sys, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
pd.set_option('display.max_columns', None)

MODEL_DIR = os.path.abspath('.')
if MODEL_DIR not in sys.path:
    sys.path.insert(0, MODEL_DIR)

import pybaseball
pybaseball.cache.enable()

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from features import (
    fetch_batting_game_log,
    build_team_game_lookup,
    build_team_stats,
    game_features,
    feature_columns,
    TEAM_NAME_TO_ABB,
    ROLLING_WINDOW,
    MIN_GAMES,
)
from train import build_training_data

ARTIFACTS_DIR = os.path.join(MODEL_DIR, 'artifacts')
MODEL_PATH = os.path.join(ARTIFACTS_DIR, 'model.pkl')
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print(f'Rolling window : {ROLLING_WINDOW} games')
print(f'Min prior games: {MIN_GAMES}')
print(f'Features       : {feature_columns()}')


## 2. Configure Training Years

In [ ]:
# 2020 is a shortened 60-game season; exclude if desired
YEARS = [2019, 2020, 2021, 2022, 2023, 2024]


## 3. Inspect a Single Team's Game Log

In [ ]:
sample_year = 2024
sample_team = 'NYY'

log = fetch_batting_game_log(sample_year, sample_team)
print(f'{sample_team} {sample_year}: {len(log)} games')
print('Columns:', log.columns.tolist())
log[['Date', 'Home', 'Opp', 'win', 'RS', 'RA', 'OBP', 'SLG', 'BB', 'SO', 'PA']].head(10)


In [ ]:
# Home vs away split
print('Home games:', log['Home'].sum())
print('Away games:', (~log['Home']).sum())
print('Win rate  :', log['win'].mean().round(3))
print('Home win% :', log[log['Home']]['win'].mean().round(3))
print('Away win% :', log[~log['Home']]['win'].mean().round(3))


## 4. Inspect Pre-Game Rolling Lookup (Single Season)

In [ ]:
# This fetches all 30 teams — takes ~3 min first run, instant after cache warms up
lookup = build_team_game_lookup(sample_year)
print(f'Teams loaded: {len(lookup)}')

nyy = lookup[sample_team]
print(f'{sample_team} columns: {nyy.columns.tolist()}')
nyy[nyy['Home']].head(10)


In [ ]:
# Plot rolling stats across the season for one team
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(nyy['Date'], nyy['rpg'],  label='RPG (offense)')
axes[0].plot(nyy['Date'], nyy['rapg'], label='RAPG (defense)')
axes[0].set_title(f'{sample_team} {sample_year} — Rolling {ROLLING_WINDOW}-game RPG / RAPG')
axes[0].legend()

axes[1].plot(nyy['Date'], nyy['obp'], label='OBP')
axes[1].plot(nyy['Date'], nyy['slg'], label='SLG')
axes[1].plot(nyy['Date'], nyy['wpct'], label='Win%', linestyle='--')
axes[1].set_title(f'{sample_team} {sample_year} — Rolling OBP / SLG / Win%')
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. Build Full Training Dataset

In [ ]:
print(f'Building training data for: {YEARS}')
X, y = build_training_data(YEARS)

print(f'\nDataset shape : {X.shape}')
print(f'Total games   : {len(X)}')
print(f'Home win rate : {y.mean():.3f}')


## 6. Feature Analysis

In [ ]:
feat_names = feature_columns()
df_feats = pd.DataFrame(X, columns=feat_names)
df_feats['home_win'] = y

print('Mean feature values by outcome:')
df_feats.groupby('home_win').mean().T.rename(columns={0: 'Away Win', 1: 'Home Win'}).round(4)


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 9))
axes = axes.flatten()
for i, col in enumerate(feat_names):
    wins   = df_feats.loc[df_feats['home_win'] == 1, col]
    losses = df_feats.loc[df_feats['home_win'] == 0, col]
    axes[i].hist(losses, bins=30, alpha=0.5, label='Away Win')
    axes[i].hist(wins,   bins=30, alpha=0.5, label='Home Win')
    axes[i].set_title(col, fontsize=9)
    axes[i].legend(fontsize=7)
plt.suptitle('Feature Distributions by Outcome', fontsize=12)
plt.tight_layout()
plt.show()


## 7. Cross-Validation

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        random_state=42,
    )),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')

print(f'CV Accuracy per fold : {scores.round(3)}')
print(f'Mean                 : {scores.mean():.3f} +/- {scores.std():.3f}')


## 8. Train Final Model

In [ ]:
pipeline.fit(X, y)

print(f'Training accuracy: {pipeline.score(X, y):.3f}')
print()
print(classification_report(y, pipeline.predict(X), target_names=['Away Win', 'Home Win']))


In [ ]:
cm = confusion_matrix(y, pipeline.predict(X))
ConfusionMatrixDisplay(cm, display_labels=['Away Win', 'Home Win']).plot()
plt.title('Confusion Matrix (Train Set)')
plt.show()


## 9. Feature Importances

In [ ]:
importances = pipeline.named_steps['model'].feature_importances_
feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)

plt.figure(figsize=(10, 4))
feat_imp.plot(kind='bar')
plt.title('Feature Importances')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(feat_imp.round(4).to_string())


## 10. Save Model

In [ ]:
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(pipeline, f)

print(f'Saved to {MODEL_PATH}')
print(f'Size: {os.path.getsize(MODEL_PATH) / 1024:.1f} KB')


## 11. Smoke Test — Predict a Matchup

In [ ]:
with open(MODEL_PATH, 'rb') as f:
    loaded_model = pickle.load(f)

current_stats = build_team_stats(datetime.now().year)
print('Current season stats (last 15 games):')
print(current_stats.round(3))


In [ ]:
home_team = 'Los Angeles Dodgers'
away_team = 'San Francisco Giants'

feat_vec = game_features(home_team, away_team, current_stats)
if feat_vec is not None:
    prob = loaded_model.predict_proba([feat_vec])[0]
    print(f'{home_team} (home) vs {away_team}')
    print(f'  Away win probability : {prob[0]:.3f}')
    print(f'  Home win probability : {prob[1]:.3f}')
else:
    print('Could not build feature vector -- check team names or data availability.')
